[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C26_Frontier_Agents_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身（MockLLM）

本课全程 **纯 numpy / 标准库、CPU 可跑、无需任何 API key**。凡是需要「模型」的地方都用 **MockLLM**——一个确定性的假模型——再用 `assert` 验证 scaffold 逻辑。

这个 notebook 做四件事：① 确认环境；② 认识 **agent = 感知-动作循环**；③ 造出本课的主角 **MockLLM**；④ 立下全课纪律——**对拍 / 不变量 + assert**。

## 1 · 环境自检

只需要标准库 + `numpy`。`matplotlib` 可选。**全程不联网、不需要 API key。**

In [ ]:
import sys, platform, json, re
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('无需 API key —— 本课用 MockLLM。环境就绪 ✅')

## 2 · agent = 感知-动作循环（最小骨架）

一次问答是 `输入→输出`。一个 **agent** 是一个**循环**：`观察 → 模型决定动作 → 环境执行 → 新观察 → ...`，直到**终止**（完成 / 超步数 / 模型声明结束）。先把这个循环的骨架写出来——用最简单的「环境」和「策略」占位。

In [ ]:
def agent_loop(policy, env, max_steps=10):
    '''最小 agent 循环骨架。
       policy(obs) -> action(dict)，约定 action['type']=='final' 表示结束。
       env(action) -> (new_obs, done)。返回 (轨迹, 最终观察)。'''
    obs = env('__reset__')[0]            # 初始观察
    trajectory = []
    for step in range(max_steps):
        action = policy(obs)             # 感知->决策
        trajectory.append((obs, action))
        if action.get('type') == 'final':
            return trajectory, action.get('answer')
        obs, done = env(action)          # 行动->新观察
        if done:
            return trajectory, obs
    return trajectory, None              # 超步数也要能停！

# 一个玩具环境：把数字累加到 >= 目标；动作 add 增加计数。
# 注意：环境本身不替策略决定『结束』——它只更新状态、永远返回 done=False，
#       何时停由策略输出 final 决定。这正是 agent『自主决定终止』的体现。
def counter_env_factory(target):
    state = {'count': 0}
    def env(action):
        if action == '__reset__':
            state['count'] = 0
            return ({'count': 0, 'target': target}, False)
        if action.get('type') == 'add':
            state['count'] += action.get('n', 1)
        return ({'count': state['count'], 'target': target}, False)
    return env

# 一个玩具策略：没到目标就 +3，到了就 final
def greedy_policy(obs):
    if obs['count'] >= obs['target']:
        return {'type': 'final', 'answer': obs['count']}
    return {'type': 'add', 'n': 3}

traj, ans = agent_loop(greedy_policy, counter_env_factory(10), max_steps=10)
print(f'循环步数={len(traj)}, 最终 count={ans}')
assert ans is not None and ans >= 10, '应当到达目标后才终止'
assert len(traj) <= 10, '必须在 max_steps 内终止'
print('✅ 感知-动作循环跑通：到达目标即终止，且永不超过 max_steps')

## 3 · 造出本课的主角：MockLLM

真实 agent 里，决定「下一步调什么工具」的是大模型。本课用 **MockLLM** 代替它：一个**确定性、规则驱动**的假模型，对给定输入返回**结构正确**的输出。这样能力的不确定性被剥离，剩下的全是我们 scaffold 的对错——最适合学协议与控制流。

下面这个 MockLLM 用「关键词→响应」规则表驱动：看到 prompt 里命中某个关键词，就返回预设的（工具调用或文本）响应。

In [ ]:
class MockLLM:
    '''确定性假模型：按规则把 prompt 映射到响应。
       规则 = [(关键词, 响应dict), ...]，第一个命中的生效；都不命中走 default。
       响应形状刻意贴近真实 tool-use：{'type':'tool_use','name','input'} 或 {'type':'text','text'}。'''
    def __init__(self, rules, default=None):
        self.rules = rules
        self.default = default or {'type': 'text', 'text': '(no rule matched)'}
        self.calls = 0                       # 记录被调次数，便于断言
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in text:
                return dict(resp)            # 返回副本，避免被调用方改坏规则
        return dict(self.default)

llm = MockLLM(rules=[
    ('天气', {'type': 'tool_use', 'name': 'get_weather', 'input': {'city': '北京'}}),
    ('你好', {'type': 'text', 'text': '你好，我能帮你查天气或算数。'}),
])
r1 = llm('请问北京今天的天气怎么样？')
r2 = llm('你好呀')
r3 = llm('随便说点啥')
print('天气问题 ->', r1)
print('打招呼   ->', r2)
print('未命中   ->', r3)
assert r1['type'] == 'tool_use' and r1['name'] == 'get_weather'
assert r2['type'] == 'text'
assert r3 == {'type': 'text', 'text': '(no rule matched)'}
assert llm.calls == 3
print('✅ MockLLM 工作正常：确定性、可断言、形状贴近真实 tool-use')

## 4 · 把 MockLLM 接进 agent 循环

现在用 MockLLM 当「策略」，跑一个最小的「问→调工具→拿结果→回答」往返。这正是模块 01 要深入的 **ReAct 回路**的雏形。

In [ ]:
# 一个被调用的「工具」
def get_weather(city):
    fake = {'北京': '晴 26°C', '上海': '多云 24°C'}
    return fake.get(city, '未知城市')

def tiny_agent(question, llm):
    '''最小 agent：第一步可能要调工具，调完把结果塞回 prompt 让模型给最终答复。'''
    resp = llm(question)
    if resp['type'] == 'tool_use' and resp['name'] == 'get_weather':
        result = get_weather(**resp['input'])         # 执行工具
        # 把工具结果回喂给模型（这里规则表里加一条命中『天气结果』）
        followup = llm(f'天气结果: {result}。请用一句话回答。')
        return followup['text'], result
    return resp.get('text', ''), None

llm2 = MockLLM(rules=[
    ('天气结果', {'type': 'text', 'text': '今天北京晴，26 度，适合出门。'}),
    ('天气',     {'type': 'tool_use', 'name': 'get_weather', 'input': {'city': '北京'}}),
])
answer, tool_out = tiny_agent('北京天气如何？', llm2)
print('工具返回 :', tool_out)
print('最终答复 :', answer)
assert tool_out == '晴 26°C'
assert '26' in answer
print('✅ 一次完整往返：问 -> 调工具 -> 拿结果 -> 回答。模块 01 把它做扎实')

## 5 · 立纪律：不变量 + assert（本课的对拍）

GPU 课用「对拍朴素实现」当裁判；agent 课的裁判是**不变量**：scaffold 在合法输入上做对、在每一类非法输入上**正确报错而非崩溃**。

先把这套工作流跑通：写一个极简的「必填字段校验」，并验证它**放行合法、拦截缺字段**——这就是模块 01 schema 校验的雏形。

In [ ]:
def check_required(args, required):
    '''检查 args 是否含 required 里的所有字段。
       返回 (ok, 缺失字段列表)。这是 schema 校验最基本的一环。'''
    missing = [k for k in required if k not in args]
    return (len(missing) == 0, missing)

ok1, miss1 = check_required({'city': '北京', 'unit': 'c'}, ['city'])
ok2, miss2 = check_required({'unit': 'c'}, ['city'])
print('合法输入 ->', ok1, miss1)
print('缺 city  ->', ok2, miss2)
assert ok1 is True and miss1 == []
assert ok2 is False and miss2 == ['city']
print('✅ 校验器：合法放行、非法精确报错 —— 这就是 agent 鲁棒性的第一道防线')

## 6 · 一个会贯穿全课的小工具：期望异常

agent 的很多「正确」表现为**正确地抛错**（未知工具、参数非法、越权调用）。把「断言某段代码确实抛了某类错」封装成一个小助手，后面每个模块都用它来验证护栏与校验确实生效。

In [ ]:
def assert_raises(fn, exc=Exception, msg=''):
    '''断言 fn() 抛出 exc 类异常；没抛或抛错类型都判失败。'''
    try:
        fn()
    except exc as e:
        return str(e)
    except Exception as e:
        raise AssertionError(f'{msg}: 期望 {exc.__name__}，却抛了 {type(e).__name__}') from None
    raise AssertionError(f'{msg}: 期望抛 {exc.__name__}，但什么都没抛')

# 演示：一个会对未知工具抛 KeyError 的迷你分发器
registry = {'get_weather': get_weather}
def dispatch(name, **kw):
    if name not in registry:
        raise KeyError(f'未知工具: {name}')   # 幻觉工具应被挡在执行之外
    return registry[name](**kw)

assert dispatch('get_weather', city='北京') == '晴 26°C'
emsg = assert_raises(lambda: dispatch('rm_rf', path='/'), KeyError, '未知工具应报错')
print('未知工具被正确拦截:', emsg)
print('✅ assert_raises 就位：本课用它验证『该报错时确实报错』')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个 scaffold（工具校验/分发、agent 循环、MCP 握手、动作解析、注入检测、pass^k）都用**不变量 + assert** 验证；逻辑正确则 assert 通过，assert 通过则可把 `MockLLM(...)` 换成真实 `messages.create(...)` 直接迁移。

**接下来六个模块**：01 工具调用 → 02 MCP → 03 computer use → 04 agentic RL → 05 评测与安全。每一步都建立在「感知-动作循环」这张图上。

下一站：**模块 01 · 结构化工具调用**。